# Lens Distortion Correction — Google Colab Training

**Before starting:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload the dataset zip to your GCS bucket: `gsutil cp automatic-lens-correction.zip gs://lens-correction/`
3. Push your code to GitHub

**Drive is used only for checkpoints and the final submission zip (~few hundred MB total — fits free tier).**
Dataset downloads from `gs://lens-correction` each session (~2-3 min).

**Handling session timeouts:** Checkpoints are saved to Drive after every epoch.
Re-run all cells to resume — training picks up from the last saved epoch automatically.

## 1. Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU
!nvidia-smi -L

In [ ]:
# Install extra deps (torch/torchvision are pre-installed on Colab)
!pip install -q timm kornia albumentations scikit-image

## 2. Configure Paths

Only edit `DRIVE_ROOT` if you want checkpoints saved somewhere other than `MyDrive/lens-correction`.

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

# Drive: only used for checkpoints + submission zip (small — fits free tier)
DRIVE_ROOT        = Path('/content/drive/MyDrive/lens-correction')
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'
DRIVE_OUTPUTS     = DRIVE_ROOT / 'outputs'

# Local Colab scratch (fast I/O, ~100 GB available, lost on session end)
LOCAL_ROOT    = Path('/content/lens-correction')
LOCAL_DATA    = LOCAL_ROOT / 'data_raw'
LOCAL_CODE    = LOCAL_ROOT / 'code'
LOCAL_CKPTS   = LOCAL_ROOT / 'checkpoints'
LOCAL_OUTPUTS = LOCAL_ROOT / 'outputs'

for p in [LOCAL_DATA, LOCAL_CODE, LOCAL_CKPTS, LOCAL_OUTPUTS, DRIVE_CHECKPOINTS, DRIVE_OUTPUTS]:
    p.mkdir(parents=True, exist_ok=True)

print('Local scratch:', LOCAL_ROOT)
print('Drive checkpoints:', DRIVE_CHECKPOINTS)

## 3. Set Up Code

**Option A** — copy from Drive (if you uploaded the project folder there)  
**Option B** — git clone from GitHub (if you pushed your code there)

In [ ]:
# Option A: copy from Drive
if DRIVE_CODE_DIR.exists():
    !cp -r {DRIVE_CODE_DIR}/* {LOCAL_CODE}/
    print('Code copied from Drive.')

# Option B: git clone (uncomment and fill in your repo URL)
# !git clone https://github.com/YOUR_USERNAME/baba-camera-correct.git {LOCAL_CODE}

sys.path.insert(0, str(LOCAL_CODE))
print('Python path set to:', LOCAL_CODE)

## 4. Download Dataset from GCS

Colab authenticates automatically via your Google account when Drive is mounted.
Run once per session (~2-3 min from GCS).

In [ ]:
import time
from google.colab import auth
auth.authenticate_user()

# ── Set your GCP project ID ────────────────────────────────────────────────────
GCP_PROJECT = 'YOUR_PROJECT_ID'   # e.g. 'my-project-123456'
!gcloud config set project {GCP_PROJECT}
# ──────────────────────────────────────────────────────────────────────────────

TRAIN_EXTRACTED = LOCAL_DATA / 'lens-correction-train-clea'
TEST_EXTRACTED  = LOCAL_DATA / 'test-originals'

if not TRAIN_EXTRACTED.exists():
    print('Downloading from gs://lens-correction (~40 GB, ~2-3 min)...')
    t0 = time.time()
    !gsutil -m cp gs://lens-correction/automatic-lens-correction.zip {LOCAL_DATA}/
    print(f'Download done in {(time.time()-t0)/60:.1f} min. Unzipping...')
    t1 = time.time()
    with zipfile.ZipFile(LOCAL_DATA / 'automatic-lens-correction.zip', 'r') as zf:
        zf.extractall(LOCAL_DATA)
    (LOCAL_DATA / 'automatic-lens-correction.zip').unlink()
    print(f'Unzip done in {(time.time()-t1)/60:.1f} min.')
else:
    print('Dataset already present, skipping download.')

train_samples = len(list(TRAIN_EXTRACTED.iterdir())) if TRAIN_EXTRACTED.exists() else 0
test_images   = len(list(TEST_EXTRACTED.glob('*.jpg'))) if TEST_EXTRACTED.exists() else 0
print(f'Train samples: {train_samples}')
print(f'Test images:   {test_images}')

## 5. Configure Paths in config.py

In [ ]:
import config

config.TRAIN_DIR       = TRAIN_EXTRACTED
config.TEST_DIR        = TEST_EXTRACTED
config.OUTPUT_DIR      = LOCAL_OUTPUTS
config.CHECKPOINT_DIR  = LOCAL_CKPTS     # local during training
config.SUBMISSION_DIR  = LOCAL_OUTPUTS / 'submissions'
config.SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

# Colab T4 has 15 GB VRAM — reduce batch sizes vs the default
config.PHASE1.batch_size = 6   # default 8
config.PHASE2.batch_size = 2   # default 4
config.PHASE2.resolution = 640  # default 768 — lower if OOM; raise for A100

print('PHASE1:', config.PHASE1)
print('PHASE2:', config.PHASE2)

## 6. Resume Helper

Copies the latest checkpoint from Drive → local before training starts,
and copies back after every epoch so progress is never lost to a session crash.

In [ ]:
def sync_checkpoints_from_drive():
    """Copy any Drive checkpoints to local before training."""
    ckpts = list(DRIVE_CHECKPOINTS.glob('*.pt'))
    if ckpts:
        for f in ckpts:
            dest = LOCAL_CKPTS / f.name
            if not dest.exists():
                shutil.copy2(f, dest)
                print(f'  Restored from Drive: {f.name}')
    else:
        print('  No Drive checkpoints found — starting fresh.')

def sync_checkpoints_to_drive():
    """Copy all local checkpoints to Drive (called after each epoch in a hook)."""
    for f in LOCAL_CKPTS.glob('*.pt'):
        dest = DRIVE_CHECKPOINTS / f.name
        shutil.copy2(f, dest)

sync_checkpoints_from_drive()

## 7. Patch Training Loop to Sync Checkpoints After Each Save

We monkey-patch `save_checkpoint` to also copy to Drive immediately.

In [ ]:
import training.train as train_module

_original_save = train_module.save_checkpoint

def _save_and_sync(*args, **kwargs):
    _original_save(*args, **kwargs)
    sync_checkpoints_to_drive()
    print('  Synced checkpoints to Drive.')

train_module.save_checkpoint = _save_and_sync
print('Checkpoint sync hook installed.')

In [ ]:
import torch
import math
import matplotlib.pyplot as plt
import cv2, numpy as np
from torch.utils.data import DataLoader
from data.dataset import LensTrainDataset
from model.lens_model import LensCorrectionModel
from losses.combined import CombinedLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Build model and loader ──────────────────────────────────────────────────
model_test = LensCorrectionModel(flow_head_active=False).to(device)
smoke_ds = LensTrainDataset(resolution=512, split='train', augment=False)
smoke_loader = DataLoader(smoke_ds, batch_size=2, shuffle=True, num_workers=2)
criterion_test = CombinedLoss(config.PHASE1_LOSS)

ok = True

# ── TEST 1: forward pass doesn't crash ─────────────────────────────────────
try:
    distorted, corrected = next(iter(smoke_loader))
    distorted, corrected = distorted.to(device), corrected.to(device)
    with torch.no_grad():
        pred, params, flow = model_test(distorted)
    print('✅ Forward pass succeeded')
    print(f'   Input shape:  {distorted.shape}')
    print(f'   Output shape: {pred.shape}')
except Exception as e:
    print(f'❌ Forward pass FAILED: {e}')
    ok = False

# ── TEST 2: loss is finite ──────────────────────────────────────────────────
if ok:
    losses_seen = []
    try:
        for i, (distorted, corrected) in enumerate(smoke_loader):
            if i >= 3: break
            distorted, corrected = distorted.to(device), corrected.to(device)
            with torch.no_grad():
                pred, params, flow = model_test(distorted)
                loss, loss_dict = criterion_test(pred, corrected, flow)
            losses_seen.append(loss.item())

        if any(math.isnan(l) or math.isinf(l) for l in losses_seen):
            print(f'❌ Loss is NaN/Inf: {losses_seen}')
            ok = False
        elif max(losses_seen) > 10.0:
            print(f'⚠️  Loss is very high ({max(losses_seen):.3f}) — may be fine at init, watch epoch 1')
        else:
            print(f'✅ Loss is finite: {[f"{l:.4f}" for l in losses_seen]}')
    except Exception as e:
        print(f'❌ Loss computation FAILED: {e}')
        ok = False

# ── TEST 3: warp is actually doing something (output ≠ input) ──────────────
if ok:
    with torch.no_grad():
        pred, params, _ = model_test(distorted)
    diff = (pred - distorted).abs().mean().item()
    if diff < 1e-6:
        print('❌ Output is identical to input — warp grid is broken')
        ok = False
    else:
        print(f'✅ Warp is active (mean pixel diff from input: {diff:.5f})')

# ── TEST 4: distortion params are in a sensible range ──────────────────────
if ok:
    s_vals = params['s'].cpu().tolist()
    k1_vals = params['k1'].cpu().tolist()
    if any(s < 0.5 or s > 2.0 for s in s_vals):
        print(f'⚠️  Scale factor s out of expected range: {s_vals}')
    else:
        print(f'✅ Distortion params look sane (s={[f"{v:.3f}" for v in s_vals]}, k1={[f"{v:.4f}" for v in k1_vals]})')

# ── TEST 5: data loading — show one pair ───────────────────────────────────
if ok:
    d_np = distorted[0].cpu().permute(1,2,0).numpy()
    p_np = pred[0].cpu().clamp(0,1).permute(1,2,0).numpy()
    c_np = corrected[0].cpu().permute(1,2,0).numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, img, title in zip(axes, [d_np, p_np, c_np], ['Distorted (input)', 'Predicted (untrained)', 'Corrected (target)']):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    plt.suptitle('Smoke test — untrained model output (should look close to distorted)', fontsize=11)
    plt.tight_layout()
    plt.show()

print()
if ok:
    print('✅ All checks passed — safe to start Phase 1 training.')
else:
    print('❌ Fix the errors above before running training.')

## 7b. Smoke Test — Run This Before Training Starts

Runs 3 batches and checks for the most common fatal flaws.
Takes ~60 seconds. If anything prints ❌ **stop and fix it before running Phase 1.**

## 8. Phase 1 — Parametric Only

Auto-resumes from the best Phase 1 checkpoint if one exists.

In [ ]:
import torch
from training.train import train_phase

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# Auto-resume if a Phase 1 checkpoint exists
p1_best = LOCAL_CKPTS / 'phase1_parametric_best.pt'
if p1_best.exists():
    config.PHASE1.resume_from = str(p1_best)
    print(f'Resuming Phase 1 from {p1_best}')
else:
    print('Starting Phase 1 from scratch.')

model = train_phase(config.PHASE1, device)

## 9. Phase 2 — Full Pipeline

Auto-resumes from best Phase 2 checkpoint, or starts from Phase 1 weights.

In [ ]:
p2_best = LOCAL_CKPTS / 'phase2_full_best.pt'
if p2_best.exists():
    config.PHASE2.resume_from = str(p2_best)
    print(f'Resuming Phase 2 from {p2_best}')
    model = None  # will be loaded from checkpoint inside train_phase
else:
    print('Starting Phase 2 from Phase 1 weights.')
    # model already holds Phase 1 weights from cell above

model = train_phase(config.PHASE2, device, model=model)

## 10. Quick Validation

In [ ]:
from torch.utils.data import DataLoader
from data.dataset import LensTrainDataset
from losses.combined import CombinedLoss
from training.validate import run_validation

val_ds = LensTrainDataset(resolution=config.PHASE2.resolution, split='val', augment=False)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
criterion = CombinedLoss(config.PHASE2_LOSS)

val_results = run_validation(
    model, val_loader, criterion, device,
    compute_proxy=True, max_proxy_samples=50,
)
print('\nValidation:')
for k, v in sorted(val_results.items()):
    print(f'  {k:30s}: {v:.4f}')

## 11. Inference on Test Images

In [ ]:
from inference.predict import run_inference

run_inference(
    checkpoint_path=str(LOCAL_CKPTS / 'phase2_full_best.pt'),
    output_dir=str(config.SUBMISSION_DIR / 'final'),
    resolution=config.PHASE2.resolution,
    do_refine=True,
    batch_size=4,
)

## 12. Package & Save Submission Zip to Drive

In [ ]:
sub_dir  = config.SUBMISSION_DIR / 'final'
zip_local = config.SUBMISSION_DIR / 'submission.zip'
zip_drive = DRIVE_OUTPUTS / 'submission.zip'

files = sorted(sub_dir.glob('*.jpg'))
with zipfile.ZipFile(zip_local, 'w', zipfile.ZIP_STORED) as zf:
    for f in files:
        zf.write(f, f.name)

# Copy to Drive so it survives session end
shutil.copy2(zip_local, zip_drive)

print(f'submission.zip: {zip_local.stat().st_size/1e6:.1f} MB, {len(files)} images')
print(f'Saved to Drive: {zip_drive}')
print('\nNext: upload submission.zip to bounty.autohdr.com')

## 13. Preview Sample Outputs

In [ ]:
import matplotlib.pyplot as plt
import cv2, numpy as np

sample_files = sorted(sub_dir.glob('*.jpg'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, f in zip(axes.flat, sample_files):
    img = cv2.imread(str(f))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f.stem[:24], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Corrected Outputs', fontsize=14)
plt.tight_layout()
plt.show()